##Imports

In [1]:
# Install the core agentic framework
!pip install -U langgraph langchain_groq joblib pandas numpy
!pip install langchain langchain-core
!pip install langchain_community
!pip install faiss-cpu sentence-transformers groq PyPDF2
!pip Install langchain-huggingface
!pip install sentence-transformers
import joblib
import pandas as pd
import numpy as np
from typing import TypedDict, List, Annotated, Union
import operator
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.4 MB/

##Model Wrapper Tool
This is the heart of your Analyser Node. It loads your .pkl files and maps the chat text to the numbers your models expect.

In [2]:
# Load all your Milestone 1 artifacts
reg_model = joblib.load('linear_model.pkl')
clf_model = joblib.load('logistic_model.pkl')
cluster_model = joblib.load('kmeans_model.pkl')
scaler_reg = joblib.load('scaler_reg.pkl')
scaler_clf = joblib.load('scaler_clf.pkl')
scaler_cluster = joblib.load('scaler_cluster.pkl')

def run_ml_pipeline(user_data: dict):
    """
    Inputs a dict of student features and returns predictions.
    Uses defaults for missing values.
    """
    # 1. Define Milestone 1 Defaults (The 'Assumptions')
    data = {
    # --- NUMERICAL DEFAULTS (MEDIANS) ---
    'MathScore': user_data.get('math', 67.0),
    'ReadingScore': user_data.get('reading', 70.0),
    'WklyStudyHours': user_data.get('study_hours', 7.5),
    'NrSiblings': user_data.get('siblings', 2.0),

    # --- RAW CATEGORICAL INPUTS WITH DEFAULTS ---
    'ParentEduc': user_data.get('parent_educ', 2),
    'PracticeSport': user_data.get('sport', 1),

    # --- ONE-HOT / BINARY ENCODINGS WITH SAFE DEFAULTS ---
    'TestPrep_none': 1 if user_data.get('test_prep', 'none') == 'none' else 0,

    'LunchType_standard': 1 if user_data.get('lunch', 'standard') == 'standard' else 0,

    'Gender_male': 1 if user_data.get('gender', 'female') == 'male' else 0,

    'IsFirstChild_yes': 1 if user_data.get('is_first_child', 'yes') == 'yes' else 0,

    'TransportMeans_school_bus': 1 if user_data.get('transport', 'school_bus') == 'school_bus' else 0
    }

    # 2. Prepare Feature Vector for Regression/Classification (11 features)
    # Order must match scaler_reg.feature_names_in_
    feat_cols = ['MathScore', 'ReadingScore', 'WklyStudyHours', 'ParentEduc',
                 'TestPrep_none', 'LunchType_standard', 'PracticeSport',
                 'NrSiblings', 'Gender_male', 'IsFirstChild_yes', 'TransportMeans_school_bus']

    X_input = pd.DataFrame([data])[feat_cols]

    # 3. Predict Exam Score & Status
    X_reg_scaled = scaler_reg.transform(X_input)
    pred_exam_score = reg_model.predict(X_reg_scaled)[0]

    X_clf_scaled = scaler_clf.transform(X_input)
    pass_fail = clf_model.predict(X_clf_scaled)[0]

    # 4. Predict Cluster (6 features: ExamScore, WklyStudyHours, ParentEduc, Lunch, TestPrep, Sport)
    cluster_data = {
        'ExamScore': pred_exam_score,
        'WklyStudyHours': data['WklyStudyHours'],
        'ParentEduc': data['ParentEduc'],
        'LunchType_standard': data['LunchType_standard'],
        'TestPrep_none': data['TestPrep_none'],
        'PracticeSport': data['PracticeSport']
    }
    cluster_cols = ['ExamScore', 'WklyStudyHours', 'ParentEduc', 'LunchType_standard', 'TestPrep_none', 'PracticeSport']
    X_cluster_input = pd.DataFrame([cluster_data])[cluster_cols]
    X_cluster_scaled = scaler_cluster.transform(X_cluster_input)

    cluster_id = cluster_model.predict(X_cluster_scaled)[0]

    # Map Cluster ID back to your M1 Labels
    # Note: Check your M1 cluster_avg to ensure this mapping matches
    category_map = {0: "At-Risk", 1: "Average", 2: "High-Performer"}

    return {
        "predicted_score": round(pred_exam_score, 2),
        "status": pass_fail,
        "category": category_map.get(cluster_id, "Unknown")
    }

##Graph State
This is the "Clipboard" that will hold the assumptions and the results.

In [29]:
from typing import Annotated, List, TypedDict
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    student_data: dict
    ml_results: dict
    retrieved_docs: List[str]
    quiz_questions: List[dict]
    current_q_idx: int
    quiz_score: int
    quiz_active: bool
    awaiting_answer: bool # NEW
    study_plan: str
    planner_notes: str
    plan: List[str]
    current_step_index: int
    next_node: str

##LLM

In [4]:
from langchain_groq import ChatGroq
import os

# Set your API Key
os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

## Extraction Logic

The biggest "Coordination Complexity" is turning a user's chat message into the dict that your run_ml_pipeline function needs. We'll create a structured extraction prompt.

In [5]:
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.messages import AIMessage, HumanMessage
import json

# --- 1. THE SCHEMA (The blueprint for data extraction) ---
class StudentDataSchema(BaseModel):
    """Structured data about a student's academic profile."""
    math: Optional[int] = Field(None, description="The math score (0-100)")
    reading: Optional[int] = Field(None, description="The reading score (0-100)")
    study_hours: Optional[float] = Field(None, description="Average weekly study hours")
    parent_educ: Optional[int] = Field(None, description="Education level: high_school:1, some_college:2, associates:3, bachelors:4, masters:5")
    test_prep: Optional[str] = Field(None, description="Test preparation: 'none' or 'completed'")
    lunch: Optional[str] = Field(None, description="Lunch type: 'standard' or 'free/reduced'")
    sport: Optional[int] = Field(None, description="Sports participation: never:0, sometimes:1, regularly:2")
    gender: Optional[str] = Field(None, description="'male' or 'female'")

# --- 2. UPDATED EXTRACTION LOGIC ---
def extraction_logic(user_message: str):
    """
    Uses structured output to guarantee valid data types.
    """
    # Force the LLM to strictly follow the StudentDataSchema
    structured_llm = llm.with_structured_output(StudentDataSchema)

    prompt = f"""
    You are a precise data extraction specialist.
    Extract values from the student's message.
    Only extract values explicitly stated or clearly implied.

    USER MESSAGE: "{user_message}"
    """

    try:
        extracted_obj = structured_llm.invoke(prompt)
        # Convert Pydantic object to dictionary
        raw_data = extracted_obj.dict()
        # CRITICAL: Filter out 'None' so we don't overwrite existing good data in the state
        clean_data = {k: v for k, v in raw_data.items() if v is not None}
        return clean_data
    except Exception as e:
        print(f"!!! Extraction Error: {e}")
        return {}

##Analyser Node
This is your first official Graph Node. It takes the state, extracts new data, merges it with existing data, and runs your ML models.

In [6]:
def analyser_node(state: AgentState):
    print("--- NODE: ANALYSER ---")

    # Get the latest human message content
    last_human_msg = [m.content for m in state["messages"] if isinstance(m, HumanMessage)][-1]

    # 1. Run the extraction
    newly_extracted_data = extraction_logic(last_human_msg)

    # 2. Update the persistent student_data in the state
    current_student_data = state.get("student_data", {}).copy()
    current_student_data.update(newly_extracted_data)

    # 3. Run the ML Pipeline with the latest merged data
    # This ensures that if they gave Math earlier and Reading now, both are used.
    ml_results = run_ml_pipeline(current_student_data)

    # 4. Construct a clear, data-driven AI response
    # We list what we captured to make the user feel "heard"
    captured_fields = ", ".join(newly_extracted_data.keys())
    if not captured_fields:
        analysis_msg = "I've reviewed your message. To give you a more accurate prediction, could you share your math or reading scores?"
    else:
        analysis_msg = (
            f"I've updated your profile with: {captured_fields}. "
            f"Your predicted exam score is {ml_results['predicted_score']} (Status: {ml_results['status']}). "
            f"You are currently in the '{ml_results['category']}' category."
        )

    return {
        "student_data": current_student_data,
        "ml_results": ml_results,
        "messages": [AIMessage(content=analysis_msg)],
        "next_node": "supervisor"
    }

## ExecutionPlan

In [7]:
from typing import Annotated, List, TypedDict, Literal
from pydantic import BaseModel, Field

# The Schema the "Architect" LLM must follow
class ExecutionPlan(BaseModel):
    steps: List[Literal["analyser", "retriever", "planner", "quizzer","end"]] = Field(
        description="A list of nodes to visit in order. Use an empty list [] for off-topic queries."
    )
    reasoning: str = Field(description="Brief explanation of why these steps were chosen.")

## The Master Node
We use a System Message to lock in the persona and a Function Call (or structured output) to handle the routing logic.

In [14]:
from typing import Annotated, List, TypedDict, Literal
import operator
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage


# ---------------- MASTER NODE ----------------
def master_node(state: AgentState):
    print("--- NODE: MASTER (ARCHITECT) ---")

    # 1. QUIZ BYPASS
    if state.get("quiz_active", False):
        return {"next_node": "quizzer"}

    plan = state.get("plan", [])
    step_idx = state.get("current_step_index", 0)

    # 2. CREATE PLAN
    if not plan or step_idx >= len(plan):

        user_msg = [m.content for m in state["messages"] if isinstance(m, HumanMessage)][-1]
        user_msg_lower = user_msg.lower()

        architect_llm = llm.with_structured_output(ExecutionPlan)

        system_prompt = f"""
You are an Agentic System Architect.

Create a minimal, correct, and efficient execution plan.

--- NODE RULES ---
- analyser → ONLY if performance/data needed
- retriever → ONLY if concept/explanation needed
- planner → ONLY if study plan requested
- quizzer → ONLY if quiz requested

--- STATE-AWARE CHECKLIST ---
Avoid unnecessary steps:
- If ml_results already exists → skip analyser
- If retrieved_docs already exists → skip retriever

--- DEPENDENCIES ---
- planner + topic → needs retriever
- planner without topic → planner only (will ask user)
- analyser before planner
- retriever before planner

--- MULTI-INTENT ---
Combine logically

--- CONSTRAINTS ---
- No duplicates
- Always end with "end"
- Off-topic → ["end"]

USER QUERY:
{user_msg}
"""

        new_plan_obj = architect_llm.invoke(system_prompt)
        new_plan = new_plan_obj.steps

        # ---------------- GUARDRAILS ----------------

        valid_nodes = {"analyser", "retriever", "planner", "quizzer", "end"}

        # remove invalid
        new_plan = [step for step in new_plan if step in valid_nodes]

        # remove duplicates
        new_plan = list(dict.fromkeys(new_plan))

        # ensure END
        if "end" not in new_plan:
            new_plan.append("end")

        # dependency fix
        if "planner" in new_plan:
            if "retriever" not in new_plan:
                if any(word in user_msg_lower for word in ["explain", "concept", "topic"]):
                    new_plan.insert(0, "retriever")

        # state-aware correction (hard rule)
        if "analyser" in new_plan and state.get("ml_results"):
            new_plan.remove("analyser")

        if "retriever" in new_plan and state.get("retrieved_docs"):
            new_plan.remove("retriever")

        # fallback
        if not new_plan:
            new_plan = ["end"]

        print(f"Plan → {new_plan}")
        print(f"Reason → {new_plan_obj.reasoning}")

        return {
            "plan": new_plan,
            "current_step_index": 1,
            "next_node": new_plan[0]
        }

    # 3. EXECUTION STEP
    next_node = plan[step_idx]

    print(f"Executing Step {step_idx+1}/{len(plan)} → {next_node}")

    return {
        "current_step_index": step_idx + 1,
        "next_node": next_node
    }

## Planner Node

This node generates a clean, structured 7-day study plan using a Pydantic schema, ensuring the output is consistent, machine-readable, and easy to convert into formats like tables, dashboards, or PDFs.

In [27]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.messages import AIMessage, HumanMessage

# Define the structure of a single day
class StudyDay(BaseModel):
    day: str = Field(description="e.g., Day 1")
    topic: str = Field(description="Focus area for the day")
    activities: List[str] = Field(description="List of specific tasks/practice sets")
    estimated_time: str = Field(description="Time required, e.g., 2 hours")

# Define the full 7-day Plan
class WeeklyPlan(BaseModel):
    title: str = Field(description="Title of the study plan")
    days: List[StudyDay] = Field(description="7 individual daily plans")
    summary: str = Field(description="Practical guidance to follow during the study plan (NOT a conclusion, NOT a completion message)")

def planner_node(state: AgentState):
    print("--- NODE: PLANNER ---")

    # 1. Gather context (ML results are optional extras)
    results = state.get('ml_results', {})
    category = results.get('category', 'Unknown')
    score = results.get('predicted_score', 'N/A')

    # 2. Get the primary intent (The Student's Interest)
    # This ensures we build a plan for what THEY asked for.
    last_user_msg = [m.content for m in state['messages'] if isinstance(m, HumanMessage)][-1]

    # 3. Pull in the RAG context (The actual "What to study")
    # Even if marks are missing, these documents provide the "Resources"
    external_knowledge = "\n".join(state.get("retrieved_docs", []))

    planner_llm = llm.with_structured_output(WeeklyPlan)

    # UPDATED PROMPT: Prioritizes Interests & RAG Resources
    prompt = f"""
    You are an expert Study Coach.

    STUDENT'S GOAL: "{last_user_msg}"
    LEARNING RESOURCES (RAG): "{external_knowledge}"

    ACADEMIC DATA (Secondary Context):
    - Category: {category}
    - Current Score: {score}

    YOUR JOB:
    Create a 7-day plan that turns the LEARNING RESOURCES into a schedule.

    CRITICAL RULES:
    1. INDEPENDENCE: If Academic Data is 'Unknown' or missing, build the plan based solely on the Goal and Resources.
    2. CONTENT: Use the specific facts and concepts found in the LEARNING RESOURCES to create the daily 'activities'.
    3. ADAPTATION: Use the Category ({category}) ONLY to tune the pace:
      - 'At-Risk': More breaks, focus on 1 concept per day from the resources.
      - 'High-Performer': Combine multiple concepts, add a "Challenge Project" on Day 7.
      - 'Unknown': Provide a standard, well-balanced study plan (do NOT mention missing data or personalization limits).
    4. NOTE: Do NOT include any message about missing data when Category is 'Unknown'. Just deliver a clean, high-quality standard plan."""

    # Generate the structured plan
    plan = planner_llm.invoke(prompt)

    # 4. Final Markdown Formatting for the UI
    markdown_plan = f"### 🗓️ {plan.title}\n"
    if category != "Unknown":
        markdown_plan += f"*Optimized for {category} performance level*\n\n"
    else:
        markdown_plan += f"*Standard Comprehensive Plan*\n\n"

    for d in plan.days:
        markdown_plan += f"**{d.day}: {d.topic}**\n"
        markdown_plan += "- " + "\n- ".join(d.activities) + f"\n*Duration: {d.estimated_time}*\n\n"

    if plan.sumary.strip():
      markdown_plan += f"---\n**How to Approach This Plan:** {plan.summary}"
    return {
        "study_plan": markdown_plan,
        "messages": [AIMessage(content=markdown_plan)],
        "next_node": "supervisor"
    }

## Retriever Node (RAG)

In [10]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from langchain_core.messages import HumanMessage

KNOWLEDGE_BASE = [

    # ── ALGEBRA ──
    "Algebra - Linear Equations: A linear equation is an equation of the form ax + b = 0. To solve it, isolate the variable by performing inverse operations (addition, subtraction, multiplication, division) on both sides equally. Example: 2x + 4 = 0 → 2x = -4 → x = -2. Key idea: maintain balance on both sides.",

    "Algebra - Quadratic Equations: A quadratic equation has the form ax² + bx + c = 0. It can be solved using factorization, completing the square, or the quadratic formula: x = (-b ± √(b² - 4ac)) / (2a). The discriminant (b² - 4ac) determines the nature of roots (real, equal, or complex).",

    "Algebra - Inequalities: Inequalities are similar to equations but involve signs like >, <, ≥, ≤. When multiplying or dividing both sides by a negative number, the inequality sign must be reversed.",

    # ── ARITHMETIC ──
    "Arithmetic - Percentages: Percentage represents parts per hundred. Formula: percentage = (part/whole) × 100. Used in profit-loss, discounts, and data interpretation problems.",

    "Arithmetic - Ratios & Proportions: A ratio compares two quantities, while proportion states equality of two ratios. Example: a/b = c/d. Used in scaling, mixtures, and real-life comparisons.",

    # ── GEOMETRY ──
    "Geometry - Triangles: The sum of interior angles of a triangle is 180°. In a right triangle, Pythagoras theorem applies: a² + b² = c². Used to find missing sides.",

    "Geometry - Circles: Key elements include radius, diameter, circumference (2πr), and area (πr²). Understanding relationships between these is essential for solving problems.",

    # ── TRIGONOMETRY ──
    "Trigonometry - Basics: Trigonometric ratios (sin, cos, tan) relate angles to sides of a right triangle. sin = opposite/hypotenuse, cos = adjacent/hypotenuse, tan = opposite/adjacent.",

    # ── STATISTICS & PROBABILITY ──
    "Statistics - Central Tendency: Mean is average, median is middle value, mode is most frequent value. Used to summarize datasets.",

    "Probability: Probability measures likelihood of an event. Formula: favorable outcomes / total outcomes. Value lies between 0 and 1.",

    # ── FUNCTIONS & CALCULUS ──
    "Functions: A function maps an input to exactly one output. Graphs help visualize relationships between variables.",

    "Calculus - Derivatives: A derivative represents rate of change (slope of a curve). Example: speed is derivative of position.",

    "Calculus - Integrals: Integrals represent accumulation or area under a curve. They are the reverse of derivatives.",

    # ── READING & COMPREHENSION ──
    "Reading Comprehension: Focus on identifying main ideas, supporting details, tone, and inference. Always connect ideas across paragraphs.",

    "Inference Skills: Inference means understanding what is implied, not directly stated. Requires combining clues from the text.",

    # ── WRITING ──
    "Writing - Structure: Use PEEL method — Point, Evidence, Explanation, Link — to construct clear and logical paragraphs.",

    # ── PROBLEM SOLVING STRATEGY ──
    "Problem Solving: First understand the problem, identify knowns and unknowns, choose a method, solve step-by-step, and verify the answer.",

    "Error Analysis: Reviewing mistakes helps identify conceptual gaps and prevents repeating errors.",

    # ── STUDY SCIENCE ──
    "Active Recall: Testing yourself without notes strengthens memory more effectively than passive review.",

    "Spaced Repetition: Revisiting topics at intervals improves long-term retention.",

    # ── RESOURCES (IMPORTANT FOR RAG RESPONSES) ──
    "For Mathematics learning, use  (khanacademy.org) for structured courses from basics to advanced.",

    "For conceptual understanding,  provides visual explanations of math topics.",

    "For advanced topics,  offers free university-level lectures.",

    "For structured courses,  allows auditing courses for free.",

    # ── META LEARNING ──
    "If you cannot explain a concept simply, you have not fully understood it.",

    "Learning builds layer by layer; weak fundamentals lead to difficulty in advanced topics.",
]

# 2. Load embedding model (local, no API)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Convert text → vectors
embeddings = embedding_model.encode(KNOWLEDGE_BASE)

# 4. Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Store mapping (important)
doc_store = {i: text for i, text in enumerate(KNOWLEDGE_BASE)}

# -----------------------------
# RETRIEVER NODE
# -----------------------------
def retriever_node(state: AgentState):
    print("--- NODE: RETRIEVER (CUSTOM RAG) ---")

    user_query = state["messages"][-1].content
    category = state.get("ml_results", {}).get("category", "General")

    # 1. Encode query
    query_vector = embedding_model.encode([user_query])

    # 2. Search FAISS
    k = 2
    distances, indices = index.search(np.array(query_vector), k)

    # 3. Get documents
    docs = [doc_store[i] for i in indices[0]]
    context = "\n".join(docs)

    # 4. LLM synthesis
    prompt = f"""
    You are an Academic Librarian.

    User Category: {category}
    User Question: {user_query}

    Retrieved Facts:
    {context}

    Instructions:
    - Answer using ONLY the retrieved facts
    - Tailor advice to the category ({category})
    - If unsure, say you don't know
    """

    response = llm.invoke(prompt)

    return {
        "retrieved_docs": docs,
        "messages": [AIMessage(content=response.content)],
        "next_node": "supervisor"
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##Quizzer Node

In [30]:
from pydantic import BaseModel, Field
from typing import List
import json
from langchain_core.messages import AIMessage, HumanMessage



# 1. Define the structure of the Quiz
class Question(BaseModel):
    question: str = Field(description="The quiz question text")
    options: List[str] = Field(description="4 multiple choice options")
    correct_answer: str = Field(description="The exact text of the correct option")
    explanation: str = Field(description="Brief explanation of why this is correct")

class QuizSet(BaseModel):
    topic: str
    questions: List[Question]

def quizzer_node(state: AgentState):
    print("--- NODE: QUIZZER ---")
    from langchain_core.messages import AIMessage

# If system just asked a question → WAIT
    if state.get("awaiting_answer", False) and isinstance(state["messages"][-1], AIMessage):
      return {"next_node": "supervisor"}

    # Get current state variables
    messages = state.get("messages", [])
    last_user_msg = None
    for m in reversed(state["messages"]):
      if isinstance(m, HumanMessage):
          last_user_msg = m.content
          break
    if not last_user_msg:
      return {"next_node": "supervisor"}
    quiz_active = state.get("quiz_active", False)
    current_idx = state.get("current_q_idx", 0)
    questions = state.get("quiz_questions", [])
    score = state.get("quiz_score", 0)

    # --- PHASE A: INITIALIZATION (Start a new quiz) ---
    if not quiz_active or not questions:
        print("Initializing New Quiz...")
        quiz_llm = llm.with_structured_output(QuizSet)

        # Use LLM to generate 3 targeted questions
        # Note: In a pro system, you could pull these from your RAG Retriever instead!
        prompt = f"Generate a 3-question MCQ quiz about: {last_user_msg}. Ensure questions match the student's level."
        generated_quiz = quiz_llm.invoke(prompt)

        return {
            "quiz_questions": [q.model_dump() for q in generated_quiz.questions],
            "current_q_idx": 0,
            "quiz_score": 0,
            "quiz_active": True,
            "awaiting_answer": True,
            "messages": [AIMessage(content=f"""Starting Quiz! Q1: {generated_quiz.questions[0].question} Options: {', '.join(generated_quiz.questions[0].options)}
👉 Reply with the exact option text OR just type A / B / C / D.""")],
            "next_node": "supervisor"
        }

    # --- PHASE B: EVALUATION (Check previous answer) ---
    current_q = questions[current_idx]

    # Simple logic: Did the user's message contain the correct answer string?
    # (In a pro version, use an LLM to grade if the answer is 'close enough')
    user_ans = last_user_msg.strip().lower()
    correct_ans = current_q['correct_answer'].strip().lower()
    # Map A/B/C/D → actual option text
    option_map = {chr(65+i).lower(): opt.lower() for i, opt in enumerate(current_q["options"])}

    if user_ans in option_map:
        user_ans = option_map[user_ans]

    is_correct = user_ans == correct_ans
    new_score = score + 1 if is_correct else score
    feedback = "✅ Correct!" if is_correct else f"❌ Incorrect. The right answer was: {current_q['correct_answer']}"
    feedback += f"\nExplanation: {current_q['explanation']}"

    # --- PHASE C: NEXT QUESTION OR END ---
    next_idx = current_idx + 1

    if next_idx < len(questions):
        # Move to next question
        next_q = questions[next_idx]
        response_text = f"""{feedback}
--- NEXT QUESTION ---
Q{next_idx+1}: {next_q['question']}
Options: {', '.join(next_q['options'])}
👉 Reply with the exact option text OR just type A / B / C / D."""

        return {
            "current_q_idx": next_idx,
            "quiz_score": new_score,
            "messages": [AIMessage(content=response_text)],
            "awaiting_answer": True,
            "next_node": "supervisor"
        }
    else:
        # Quiz Finished
        final_msg = f"{feedback}\n\n🏆 **Quiz Finished!**\nYour Final Score: {new_score}/{len(questions)}"
        return {
            "quiz_active": False,
            "awaiting_answer": False,
            "quiz_score": new_score,
            "messages": [AIMessage(content=final_msg)],
            "next_node": "supervisor"
        }

##End/Response Node

In [12]:
from langchain_core.messages import AIMessage
def end_node(state: AgentState):
    print("--- NODE: END/RESPONSE ---")

    last_message = state["messages"][-1].content

    # Simple logic to determine the tone of the final response
    prompt = f"""
    You are an AI Study Coach generating the FINAL response.

    USER MESSAGE:
    "{last_message}"

    AVAILABLE CONTEXT:
    - Student Data: {state.get("student_data")}
    - ML Analysis: {state.get("ml_results")}
    - Retrieved Knowledge: {state.get("retrieved_docs")}
    - Study Plan: {state.get("study_plan")}
    - Planner Notes: {state.get("planner_notes")}
    - Quiz Active: {state.get("quiz_active")}

    --- CORE PRINCIPLE ---
    Select and combine ONLY relevant information.
    Do NOT blindly merge all available context.

    --- STEP 1: DETECT INTENTS ---
    Possible intents:
    - GREETING
    - CLOSING
    - STUDY_PLAN
    - CONCEPT_EXPLANATION
    - PERFORMANCE_INSIGHT
    - QUIZ
    - OFF_TOPIC

    Multiple intents may exist in one query.

    --- STEP 2: PRIORITIZATION ---
    If multiple intents are present, follow priority:
    1. QUIZ (highest)
    2. STUDY_PLAN
    3. PERFORMANCE_INSIGHT
    4. CONCEPT_EXPLANATION
    5. GREETING / CLOSING
    6. OFF_TOPIC (lowest)

    --- STEP 3: CONTEXT SELECTION ---
    - STUDY_PLAN → study_plan + planner_notes
    - CONCEPT_EXPLANATION → retrieved_docs
    - PERFORMANCE_INSIGHT → ml_results (+ student_data if needed)
    - QUIZ → quiz context only
    - GREETING / CLOSING → no extra context
    - OFF_TOPIC → redirect only

    --- STEP 4: RESPONSE STRUCTURE (IMPORTANT) ---

    If SINGLE intent:
    → Respond normally (clean and focused)

    If MULTIPLE intents:
    → Use structured sections in this order:

    1. 📊 Performance Insight (if present)
    2. 📚 Concept Explanation (if present)
    3. 🗓️ Study Plan (if present)

    --- FORMAT RULES ---
    - Use clear section headers ONLY when multiple intents exist
    - Do NOT include irrelevant sections
    - Keep flow natural (not robotic)
    - Never dump raw data — always explain

    --- LENGTH CONTROL ---
    - Simple queries → short
    - Multi-intent / plan → detailed but structured

    --- STYLE ---
    - Supportive, clear, confident
    - No internal system terms
    - No unnecessary filler

    --- EDGE CASES ---
    - If required context is missing → give best general answer
    - If user intent is unclear → assume most academic intent

    Now generate the best possible response.
    """

    response = llm.invoke(prompt)

    # We return the message, but we DON'T set a 'next_node'
    # because this is the termination point of the graph.
    return {
        "messages": [AIMessage(content=response.content)]
    }

### Compiling the LangGraph
Now that all your "Departments" (Nodes) are built, we need to connect them. This is where we define the Edges.

In [15]:
from langgraph.graph import StateGraph, END

# 1. Initialize the Graph with our State definition
workflow = StateGraph(AgentState)

# 2. Add all our Nodes
workflow.add_node("supervisor", master_node)
workflow.add_node("analyser", analyser_node)
workflow.add_node("planner", planner_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("quizzer", quizzer_node)
workflow.add_node("end", end_node)

# 3. Define the Edges (The "Flow")
# The Supervisor is the Entry Point
workflow.set_entry_point("supervisor")

# The Supervisor decides where to go based on 'next_node'
workflow.add_conditional_edges(
    "supervisor",
    lambda x: x["next_node"],
    {
        "analyser": "analyser",
        "planner": "planner",
        "retriever": "retriever",
        "quizzer": "quizzer",
        "end": "end"
    }
)

# Every specialist node goes BACK to the supervisor to see if more is needed
workflow.add_edge("analyser", "supervisor")
workflow.add_edge("planner", "supervisor")
workflow.add_edge("retriever", "supervisor")
workflow.add_edge("quizzer", "supervisor")

# The 'end' node leads to the actual END of the graph
workflow.add_edge("end", END)

# 4. Compile the Graph
app = workflow.compile()
print("Graph Compiled Successfully!")

Graph Compiled Successfully!


In [16]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Explain algebra")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['retriever', 'end']
Reason → The user is asking for an explanation of algebra, which requires a concept/explanation. Therefore, the retriever step is necessary. Since there are no mentions of performance/data, study plan, or quiz, the analyser, planner, and quizzer steps are not required.
--- NODE: RETRIEVER (CUSTOM RAG) ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 2/2 → end
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

Explain algebra
Algebra can be explained through the concept of linear equations. A linear equation is of the form ax + b = 0. To solve it, you need to isolate the variable by performing inverse operations, such as addition, subtraction, multiplication, or division, on both sides equally. 

For example, let's take the equation 2x + 4 = 0. To solve for x, you would first subtract 4 from both sides, resulting in 2x = -4. Then, you would divide both sides by 2, giving you x = -2. The key idea here is to maintain balance o

In [28]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Give me a study plan to improve my math")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['retriever', 'planner', 'end']
Reason → The user is requesting a study plan to improve their math skills. This requires the planner step. Since the topic is specified (math), the retriever step is also necessary before the planner step. The analyser step is not required as there is no mention of performance or data needs. The quizzer step is also not required as there is no quiz request.
--- NODE: RETRIEVER (CUSTOM RAG) ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 2/3 → planner
--- NODE: PLANNER ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 3/3 → end
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

Give me a study plan to improve my math
To improve your math, I recommend using Khan Academy (khanacademy.org) for structured courses that cover topics from basics to advanced levels. Additionally, Khan Academy provides visual explanations of math topics, which can help with conceptual understanding. This should give you a good foundatio

In [32]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Give me a quiz on algebra")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['retriever', 'quizzer', 'end']
Reason → The user requested a quiz on algebra, so we need to use the quizzer. Since the quizzer requires a topic, we also need to use the retriever to get the relevant information on algebra. The planner is not needed in this case because the user did not request a study plan. The analyser is also not needed because the user did not ask for performance or data. Therefore, the execution plan should include the retriever and the quizzer.
--- NODE: RETRIEVER (CUSTOM RAG) ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 2/3 → quizzer
--- NODE: QUIZZER ---
Initializing New Quiz...


/tmp/ipykernel_9531/1438320165.py:39: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  "quiz_questions": [q.dict() for q in generated_quiz.questions],


--- NODE: MASTER (ARCHITECT) ---
--- NODE: QUIZZER ---
--- NODE: MASTER (ARCHITECT) ---
--- NODE: QUIZZER ---
--- NODE: MASTER (ARCHITECT) ---
--- NODE: QUIZZER ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 3/3 → end
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

Give me a quiz on algebra
I'd be happy to give you a quiz on algebra. Here's your first question:

What is the formula used to solve a quadratic equation of the form ax² + bx + c = 0?

A) x = (-b + √(b² - 4ac)) / (2a)
B) x = (-b ± √(b² - 4ac)) / (2a)
C) x = (-b - √(b² - 4ac)) / (2a)
D) x = (b ± √(b² - 4ac)) / (2a)

Let me know your answer.

Also, I can give you a question on inequalities if you'd like. For example, what happens when you multiply or divide both sides of an inequality by a negative number?
Starting Quiz! Q1: What is the formula used to solve a quadratic equation of the form ax² + bx + c = 0?
Options: x = (-b + √(b² - 4ac)) / (2a), x = (-b ± √(b² - 4ac)) / (2a), x = (-b - √(b² - 4ac)) / (2a), x = (b ± √(

In [19]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Hi")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['end']
Reason → No specific intent or topic mentioned in the user query, so no specific steps are required. The conversation just started, so we can only end it as there's no information to process.
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

Hi
Hello. It's nice to meet you. I'm here to help you with your studies. What would you like to focus on today?


In [20]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="I scored 42 in math and study only 1 hour daily. Am I performing well or not?")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['analyser', 'end']
Reason → The user is asking about their performance in math, which requires analysis of their score and study habits. Since performance/data is needed, the analyser step is required. The retriever step is not necessary as the user is not asking for a concept or explanation. The planner step is also not needed as the user is not requesting a study plan. The quizzer step is not required as the user is not asking for a quiz.
--- NODE: ANALYSER ---


/tmp/ipykernel_9531/2532591600.py:37: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  raw_data = extracted_obj.dict()


--- NODE: MASTER (ARCHITECT) ---
Executing Step 2/2 → end
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

I scored 42 in math and study only 1 hour daily. Am I performing well or not?
I've updated your profile with: math, study_hours. Your predicted exam score is 63.19 (Status: Fail). You are currently in the 'Average' category.
Based on your updated profile, I've analyzed your data and here's what I found: 

📊 Performance Insight
Your predicted exam score is 63.19, which unfortunately indicates that you're currently at risk of not passing. This places you in the 'Average' category. To improve, let's take a closer look at your study habits and math skills. With only 1 hour of study time reported and a math score of 42, it's clear that we need to work on increasing your study hours and focusing on math concepts.

Since you're in the 'Average' category, we can create a tailored study plan to help you improve. However, I'd like to know more about your goals and what you'd like to achiev

In [21]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="My math score is 48. Explain quadratic equations and give me a study plan to improve.")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['retriever', 'planner', 'end']
Reason → The user needs an explanation of quadratic equations and a study plan to improve their math score. Since the user's score is provided, the analyser is not necessary. The retriever is needed to explain quadratic equations, and the planner is required to create a study plan. The quizzer is not requested.
--- NODE: RETRIEVER (CUSTOM RAG) ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 2/3 → planner
--- NODE: PLANNER ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 3/3 → end
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

My math score is 48. Explain quadratic equations and give me a study plan to improve.
As a general student, I'd like to explain quadratic equations in a simple way. A quadratic equation has the form ax² + bx + c = 0. To solve it, you can use factorization, completing the square, or the quadratic formula: x = (-b ± √(b² - 4ac)) / (2a). The discriminant (b² - 4ac) helps determine the n

In [34]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="I got around 52 in my math exam. I'm struggling with quadratic equations. Can you explain it in a simple way, suggest how I should study, and maybe give me a small quiz to practice quadratic equations?")]    ,
    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['retriever', 'planner', 'quizzer', 'end']
Reason → The user needs an explanation of quadratic equations, a study plan, and a quiz. Since the user is struggling with quadratic equations, we need to provide an explanation first, which requires the retriever. Then, we can provide a study plan using the planner, and finally, a quiz using the quizzer. The analyser is not necessary in this case since the user is not asking for performance/data analysis.
--- NODE: RETRIEVER (CUSTOM RAG) ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 2/4 → planner
--- NODE: PLANNER ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 3/4 → quizzer
--- NODE: QUIZZER ---
Initializing New Quiz...


/tmp/ipykernel_9531/1438320165.py:39: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  "quiz_questions": [q.dict() for q in generated_quiz.questions],


--- NODE: MASTER (ARCHITECT) ---
--- NODE: QUIZZER ---
--- NODE: MASTER (ARCHITECT) ---
--- NODE: QUIZZER ---
--- NODE: MASTER (ARCHITECT) ---
--- NODE: QUIZZER ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 4/4 → end
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

I got around 52 in my math exam. I'm struggling with quadratic equations. Can you explain it in a simple way, suggest how I should study, and maybe give me a small quiz to practice quadratic equations?
I'd be happy to help you with quadratic equations. A quadratic equation has the form ax² + bx + c = 0. To solve it, you can use factorization, completing the square, or the quadratic formula: x = (-b ± √(b² - 4ac)) / (2a). 

To study, I suggest you start by understanding the concept of linear equations, which is an equation of the form ax + b = 0. You can solve it by isolating the variable by performing inverse operations on both sides equally. Then, move on to quadratic equations and practice solving them using the dif

In [35]:
from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="I scored 52 in math. How am I doing and what should I do to improve?")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],
    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "study_plan": "",
    "planner_notes": "",
    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

--- NODE: MASTER (ARCHITECT) ---
Plan → ['analyser', 'retriever', 'planner', 'end']
Reason → The user is asking for their performance in math and how to improve, which requires analysis and a study plan. Since the user's score is provided, we can skip the analyser step if ml_results already exists. However, to provide a study plan, we need to retrieve relevant documents and then plan. Therefore, the execution plan should include retriever and planner steps. The quizzer step is not necessary as the user is not requesting a quiz.
--- NODE: ANALYSER ---


/tmp/ipykernel_9531/2532591600.py:37: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  raw_data = extracted_obj.dict()


--- NODE: MASTER (ARCHITECT) ---
Executing Step 2/4 → retriever
--- NODE: RETRIEVER (CUSTOM RAG) ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 3/4 → planner
--- NODE: PLANNER ---
--- NODE: MASTER (ARCHITECT) ---
Executing Step 4/4 → end
--- NODE: END/RESPONSE ---

--- FINAL OUTPUT ---

I scored 52 in math. How am I doing and what should I do to improve?
I've updated your profile with: math. Your predicted exam score is 66.09 (Status: Fail). You are currently in the 'Average' category.
Given your predicted exam score of 66.09, which falls under the 'Fail' status, it seems you need to improve your understanding of the subjects, specifically math. 

As an Average category student, focusing on the basics is crucial. Let's start with the concept of probability. It measures the likelihood of an event, calculated using the formula: favorable outcomes divided by total outcomes. The value of probability always lies between 0 and 1. Understanding this concept can help you solve problems re

In [ ]:
# # 1. Setup a state that looks like a user just gave a score
# test_analyser_state = {
#     "messages": [HumanMessage(content="I got a 20 in Math and 15 in Reading.")],
#     "student_data": {},
#     "ml_results": {},
#     "quiz_active": False
# }

# print("--- TESTING INDIVIDUAL NODE: ANALYSER ---")

# # 2. Call the node function DIRECTLY (bypassing the graph)
# analyser_output = analyser_node(test_analyser_state)

# # 3. Verification
# print("1. Did it update student_data?", analyser_output.get("student_data"))
# print("2. Did it run ML and get a category?", analyser_output.get("ml_results").get("category"))
# print("3. Is the message an AIMessage?", isinstance(analyser_output["messages"][0], AIMessage))
# print("4. Response Text:", analyser_output["messages"][0].content)

In [ ]:
# # Test Case: The Planner Node
# # We use the state created by your successful Analyser test
# test_planner_state = {
#     "messages": [HumanMessage(content="Make me a study plan.")],
#     "student_data": {'math': 20, 'reading': 15},
#     "ml_results": {'predicted_score': 20.01, 'status': 'Fail', 'category': 'Average'}, # Using your result
#     "quiz_active": False
# }

# print("--- TESTING INDIVIDUAL NODE: PLANNER ---")

# # 1. Call the Planner node directly
# planner_output = planner_node(test_planner_state)

# # 2. Verification
# print("1. Did it generate a Plan?", "Yes" if planner_output.get("study_plan") else "No")
# print("2. Is the message an AIMessage?", isinstance(planner_output["messages"][0], AIMessage))
# print("-" * 30)
# print("AI Response Text:\n", planner_output["messages"][0].content)

In [ ]:
# # Test Case: The Retriever Node
# # We want to see if it finds the specific quadratic formula from your materials.
# test_retriever_state = {
#     "messages": [HumanMessage(content="What is the formula for quadratic equations and how do I use it?")],
#     "student_data": {'math': 20, 'reading': 15},
#     "ml_results": {'predicted_score': 20.01, 'status': 'Fail', 'category': 'Average'},
#     "quiz_active": False,
#     "retrieved_docs": []
# }

# print("--- TESTING INDIVIDUAL NODE: RETRIEVER ---")

# # 1. Call the Retriever node directly
# retriever_output = retriever_node(test_retriever_state)

# # 2. Verification
# print(f"1. Documents found in FAISS: {len(retriever_output.get('retrieved_docs', []))}")
# print("-" * 30)
# print("AI Response Text:\n", retriever_output["messages"][0].content)

In [ ]:
# # -----------------------------
# # INITIAL STATE (same as yours)
# # -----------------------------
# state = {
#     "messages": [HumanMessage(content="Quiz me on Algebra.")],
#     "student_data": {'math': 20, 'reading': 15},
#     "ml_results": {'predicted_score': 20.01, 'status': 'Fail', 'category': 'Average'},
#     "quiz_active": False,
#     "quiz_questions": [],
#     "current_q_idx": 0,
#     "quiz_score": 0
# }


# # -----------------------------
# # START QUIZ (Phase 1)
# # -----------------------------
# output = quizzer_node(state)

# # Merge state
# state.update(output)

# print("\n🧠 QUIZ STARTED")
# print(state["messages"][0].content)


# # -----------------------------
# # INTERACTIVE LOOP
# # -----------------------------
# while state.get("quiz_active", False):

#     # Take user input
#     user_answer = input("\nYour Answer: ")

#     # Append answer to message history
#     state["messages"].append(HumanMessage(content=user_answer))

#     # Call your quizzer node again
#     output = quizzer_node(state)

#     # Update state
#     state.update(output)

#     # Print AI response (feedback + next question)
#     print("\n" + state["messages"][0].content)


# print("\n✅ Quiz Completed!")
# print(f"Final Score: {state.get('quiz_score')}")

# Router Test Master node routing test

In [ ]:
# # Test 1: Data Extraction Routing
# test_state = {
#     "messages": [HumanMessage(content="I got an 85 in Math and a 90 in Reading.")],
#     "quiz_active": False,
#     "student_data": {},
#     "ml_results": {}
# }

# print("--- TESTING SUPERVISOR: EXTRACTION ---")

# # 1. Call the supervisor node
# result = supervisor_node(test_state)

# # 2. Verification
# print(f"Query: 'I got an 85 in Math and a 90 in Reading.'")
# print(f"Next Node: {result.get('next_node')}")

# # Expected: 'analyser'

In [ ]:
# # Test 2: Knowledge Retrieval Routing
# test_state = {
#     "messages": [HumanMessage(content="Can you explain how photosynthesis works?")],
#     "quiz_active": False,
#     "student_data": {'math': 85}, # Data already exists
#     "ml_results": {'status': 'Pass'}
# }

# print("--- TESTING SUPERVISOR: KNOWLEDGE/RAG ---")

# # 1. Call the supervisor node
# result = supervisor_node(test_state)

# # 2. Verification
# print(f"Query: 'Can you explain how photosynthesis works?'")
# print(f"Next Node: {result.get('next_node')}")

# # Expected: 'retriever'

In [ ]:
# # Test 3: Quiz Trigger Routing
# test_state = {
#     "messages": [HumanMessage(content="Can you give me a quiz on math?")],
#     "quiz_active": False,
#     "student_data": {'math': 85},
#     "ml_results": {'status': 'Pass'}
# }

# print("--- TESTING SUPERVISOR: QUIZ TRIGGER ---")

# # 1. Call the supervisor node
# result = supervisor_node(test_state)

# # 2. Verification
# print(f"Query: 'Can you give me a quiz on math?'")
# print(f"Next Node: {result.get('next_node')}")

# # Expected: 'quizzer'

In [ ]:
# # Test 4: Quiz Lockdown Routing
# # Even if the user asks a question about photosynthesis,
# # if a quiz is active, we MUST stay in the quizzer node.
# test_state = {
#     "messages": [
#         HumanMessage(content="Can you explain photosynthesis?") # A distractor query
#     ],
#     "quiz_active": True, # THE LOCK
#     "student_data": {'math': 85},
#     "ml_results": {'status': 'Pass'}
# }

# print("--- TESTING SUPERVISOR: QUIZ LOCKDOWN ---")

# # 1. Call the supervisor node
# result = supervisor_node(test_state)

# # 2. Verification
# print(f"Query: 'Can you explain photosynthesis?' (while Quiz is Active)")
# print(f"Next Node: {result.get('next_node')}")

# # Expected: 'quizzer'

In [ ]:
# # Test 5: Planner Routing
# # Goal: Ensure the supervisor routes to 'planner' when a study schedule is requested.
# test_state = {
#     "messages": [HumanMessage(content="Can you create a 7-day study plan for me?")],
#     "quiz_active": False,
#     "student_data": {'math': 95, 'reading': 88},
#     "ml_results": {'predicted_score': 92.5, 'status': 'Pass', 'category': 'High-Performer'}
# }

# print("--- TESTING SUPERVISOR: PLANNER ROUTING ---")

# # 1. Call the supervisor node
# result = supervisor_node(test_state)

# # 2. Verification
# print(f"Query: 'Can you create a 7-day study plan for me?'")
# print(f"Next Node: {result.get('next_node')}")

# # Expected: 'planner'

In [ ]:
# # FINAL END-TO-END INTEGRATION TEST
# # Verifying that State persists across multiple node transitions.
# from langchain_core.messages import HumanMessage, AIMessage

# print("--- STARTING FINAL INTEGRATION TEST ---")

# # 1. Helper to simulate the Graph's movement
# def run_step(query, current_state):
#     # Add user message to history
#     current_state["messages"].append(HumanMessage(content=query))

#     # Let Supervisor decide
#     decision = supervisor_node(current_state)
#     target = decision["next_node"]

#     print(f"\n[USER]: {query}")
#     print(f"[ROUTING]: -> {target}")

#     # Execute the chosen node
#     if target == "analyser":
#         res = analyser_node(current_state)
#     elif target == "planner":
#         res = planner_node(current_state)
#     elif target == "retriever":
#         res = retriever_node(current_state)
#     elif target == "quizzer":
#         res = quizzer_node(current_state)
#     else:
#         print("End of flow.")
#         return current_state

#     # Update the global state with results from the node
#     current_state.update(res)

#     # Print a snippet of the AI response
#     print(f"[AI]: {res['messages'][0].content[:150]}...")
#     return current_state

# # 2. Initialize fresh state
# session_state = {
#     "messages": [],
#     "student_data": {},
#     "ml_results": {},
#     "quiz_active": False,
#     "quiz_questions": [],
#     "current_q_idx": 0,
#     "quiz_score": 0
# }

# # --- THE JOURNEY ---

# # Step 1: Profile Creation (Analyser)
# # Verifies that data extraction and ML pipeline work together.
# session_state = run_step("Hi, I'm a male student. I got 95 in Math and 82 in Reading.", session_state)

# # Step 2: Personalized Planning (Planner)
# # Verifies that the Planner reads the 'High-Performer' category from Step 1.
# session_state = run_step("Can you make a study plan for me?", session_state)

# # Step 3: Specific Support (Retriever)
# # Verifies that we can pivot to knowledge retrieval without losing session data.
# session_state = run_step("I also need the quadratic formula.", session_state)

# # Step 4: Assessment (Quizzer Init)
# # Verifies that the Quizzer triggers and locks the session.
# session_state = run_step("Actually, test me on Algebra now.", session_state)

# # Step 5: Quiz Response (Quizzer Evaluation)
# # Verifies that the Supervisor locks the user in the Quizzer while quiz_active is True.
# if session_state.get("quiz_active"):
#     session_state = run_step("I think the answer is 3.", session_state)

# print("\n" + "="*40)
# print("INTEGRATION CHECKLIST:")
# print(f"1. Student Data Saved?      {'math' in session_state['student_data']} (Score: {session_state['student_data'].get('math')})")
# print(f"2. ML Results Persisted?    {'predicted_score' in session_state['ml_results']} (Category: {session_state['ml_results'].get('category')})")
# print(f"3. Conversation Context?    {len(session_state['messages'])} messages stored")
# print(f"4. Quiz State Maintained?   {session_state.get('quiz_active') or 'Completed'}")
# print("="*40)
# print("\n✅ INTEGRATION TEST SUITE READY")

In [ ]:
# def run_autonomous_agent(initial_query):
#     # 1. Initialize State
#     current_state = {
#         "messages": [HumanMessage(content=initial_query)],
#         "student_data": {},
#         "ml_results": {},
#         "quiz_active": False,
#         "next_node": "supervisor",
#         "history_log": [] # Internal tracker to prevent loops
#     }

#     # 2. The Graph Loop (Simulating LangGraph's internal engine)
#     steps_taken = 0
#     max_steps = 10

#     while steps_taken < max_steps:
#         # Ask Supervisor: "What's next?"
#         decision = supervisor_node(current_state)
#         next_step = decision["next_node"]

#         # Guard against infinite loops on the same node
#         if steps_taken > 0 and next_step == current_state["history_log"][-1] and next_step != "quizzer":
#             print(f"⚠️ Loop detected on node: {next_step}. Forcing transition.")
#             if next_step == "analyser":
#                 next_step = "planner"
#             else:
#                 next_step = "end"

#         if next_step == "end":
#             print("\n🏁 Agent reached 'END'. Goal accomplished.")
#             break

#         print(f"\nStep {steps_taken + 1}: Supervisor routing to -> {next_step}")
#         current_state["history_log"].append(next_step)

#         # Call the actual node
#         if next_step == "analyser":
#             output = analyser_node(current_state)
#         elif next_step == "planner":
#             output = planner_node(current_state)
#         elif next_step == "retriever":
#             output = retriever_node(current_state)
#         elif next_step == "quizzer":
#             output = quizzer_node(current_state)

#         # --- UPDATE STATE ---
#         if "messages" in output:
#             # Print the response so you can see it!
#             new_msg = output["messages"][0].content
#             print(f"[{next_step.upper()} RESPONSE]:\n{new_msg}")

#             current_state["messages"].extend(output["messages"])

#         # Update other state keys (student_data, ml_results, etc.)
#         for key, value in output.items():
#             if key != "messages":
#                 current_state[key] = value

#         steps_taken += 1

#     return current_state

# # --- THE ULTIMATE TEST ---
# print("--- RUNNING AUTONOMOUS MULTI-STEP TEST ---")
# final_result = run_autonomous_agent("I'm a male student with 90 in math. Analyze this and give me a study plan.")

In [ ]:

# print("🚀 LAUNCHING AUTONOMOUS TEST VIA APP.INVOKE()")
# print("QUERY: 'I got a 75 in Science. Explain Photosynthesis and make a plan.'")

# final_state = app.invoke({
#     "messages": [HumanMessage(content="I got a 75 in Science. Explain Photosynthesis and make a plan.")],
#     "student_data": {},
#     "ml_results": {},
#     "quiz_active": False,
#     "study_plan": ""
# })

# print("\n" + "="*50)
# print("FINAL CONVERSATION LOG:")
# for msg in final_state["messages"]:
#     role = "USER" if isinstance(msg, HumanMessage) else "AI"
#     print(f"{role}: {msg.content}")
# print("="*50)
# print("✅ TEST COMPLETE: The agent autonomously navigated the chain and stopped at END.")